In [ ]:
import sys
import os
import re
import pandas as pd
import numpy as np

if not os.path.exists("pyproject.toml"):
    os.chdir('../../')
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.enrichment.engine import EnrichmentEngine
from tests.enrichment.utils import ENGINE, LAYOUTS

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

df_full = pd.read_parquet('data/interim/normalized_sequences.parquet')
sample_ids = df_full['SEQUENCE_ID'].drop_duplicates().sample(50000, random_state=42)
df_sample = df_full[df_full['SEQUENCE_ID'].isin(sample_ids)]

def reconstruct_text(group):
    return "".join([chr(k) for k in group['KEY_ID'].values if k != 0])

seq_texts = df_sample.groupby('SEQUENCE_ID').apply(reconstruct_text)

# ---------------------------------------------------------
target_words = ["cat","these","tomorrow"]
target_features = ["is_syllable_start","is_syllable_end"]
layouts =["qwerty", "dvorak"]    
max_examples = 5
show_only_target_features = True 
# ---------------------------------------------------------

pattern = '|'.join(target_words)
matches = seq_texts[seq_texts.str.contains(pattern, regex=True, case=False)]

if matches.empty:
    print("No matches found.")
else:
    for count, target_id in enumerate(matches.index[:max_examples]):
        full_text = matches[target_id]
        
        match_obj = re.search(pattern, full_text, flags=re.IGNORECASE)
        start_idx = match_obj.start()
        matched_str = match_obj.group()
        
        print(f"\nExample {count+1}: Found '{matched_str}' in Sequence: {target_id}")
        print(f"Start index: {start_idx}")
        print(f"Full Text preview: {full_text[:50]}...\n")
        
        df_target = df_full[df_full['SEQUENCE_ID'] == target_id].copy()
        
        enriched_preview, key_ids = ENGINE.enrich(df_target, n_pads=0, **LAYOUTS[layouts[0]])
        
        preview_data = {'key':[chr(i) if i != 0 else "[PAD]" for i in key_ids]}
        
        for k, v in enriched_preview.items():
            if show_only_target_features and k not in target_features:
                continue
            if k in['finger', 'finger_type']:
                preview_data[k] = np.where(v[:, 0] == -1.0, -1, np.argmax(v, axis=1))
            elif isinstance(v, np.ndarray) and len(v.shape) > 1:
                for col_idx in range(v.shape[1]):
                    preview_data[f"{k}_{col_idx}"] = v[:, col_idx]
            else:
                preview_data[k] = v
                
        preview_df = pd.DataFrame(preview_data)
        word_preview = preview_df.iloc[start_idx:start_idx+len(matched_str)]
        
        print(f"--- DataFrame Preview ({layouts[0]} layout) ---")
        display(word_preview.T)
        
        print("\n--- AUTO-GENERATED TEST CODE ---")
        print(f"# {matched_str.lower()}")
        print(f"df_real = load_real_data(['{target_id}'])")
        
        for feat in target_features:
            for layout in layouts:
                enriched_eval, _ = ENGINE.enrich(df_target, n_pads=0, **LAYOUTS[layout])
                
                if feat not in enriched_eval:
                    continue
                    
                test_vals = enriched_eval[feat][start_idx:start_idx+len(matched_str)]
                is_ohe = feat in['finger', 'finger_type']
                
                if is_ohe:
                    test_vals = np.where(test_vals[:, 0] == -1.0, -1, np.argmax(test_vals, axis=1))
                
                if len(np.shape(test_vals)) == 1:
                    can_str = True
                    s_out = ""
                    l_out =[]
                    for v in test_vals:
                        if pd.isna(v): s_out += "N"; l_out.append("np.nan")
                        elif v == -1: s_out += "P"; l_out.append("-1.0")
                        elif isinstance(v, (int, float, np.integer, np.floating)) and v == int(v) and 0 <= v <= 9:
                            s_out += str(int(v)); l_out.append(str(float(v)))
                        else:
                            can_str = False; l_out.append(str(v))
                    exp_repr = f'"{s_out}"' if can_str else f"[{', '.join(l_out)}]"
                else:
                    l_out =[]
                    for row in test_vals:
                        row_out = ["np.nan" if pd.isna(v) else str(v) for v in row]
                        l_out.append("[" + ", ".join(row_out) + "]")
                    exp_repr = "[" + ", ".join(l_out) + "]"
                    
                print(f"tester_{feat}.check_real(df_real, expected={exp_repr}, start_idx={start_idx}, layout='{layout}')")
            print()
            
        print("="*80)


Example 1: Found 'cat' in Sequence: 100446_1095949_0_1
Start index: 4
Full Text preview: 'll catch...

--- DataFrame Preview (qwerty layout) ---


,4,5,6
key,c,a,t
is_syllable_start,1.0,0.0,0.0
is_syllable_end,0.0,0.0,0.0



--- AUTO-GENERATED TEST CODE ---
# cat
df_real = load_real_data(['100446_1095949_0_1'])
tester_is_syllable_start.check_real(df_real, expected="100", start_idx=4, layout='qwerty')
tester_is_syllable_start.check_real(df_real, expected="100", start_idx=4, layout='dvorak')

tester_is_syllable_end.check_real(df_real, expected="000", start_idx=4, layout='qwerty')
tester_is_syllable_end.check_real(df_real, expected="000", start_idx=4, layout='dvorak')


Example 2: Found 'tomorrow' in Sequence: 100581_1097630_2_0
Start index: 2
Full Text preview: n tomorrow....

--- DataFrame Preview (qwerty layout) ---


,2,3,4,5,6,7,8,9
key,t,o,m,o,r,r,o,w
is_syllable_start,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
is_syllable_end,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0



--- AUTO-GENERATED TEST CODE ---
# tomorrow
df_real = load_real_data(['100581_1097630_2_0'])
tester_is_syllable_start.check_real(df_real, expected="10010100", start_idx=2, layout='qwerty')
tester_is_syllable_start.check_real(df_real, expected="10010100", start_idx=2, layout='dvorak')

tester_is_syllable_end.check_real(df_real, expected="00101001", start_idx=2, layout='qwerty')
tester_is_syllable_end.check_real(df_real, expected="00101001", start_idx=2, layout='dvorak')


Example 3: Found 'these' in Sequence: 101030_1102308_0_0
Start index: 21
Full Text preview: In isolation, all of these things ...

--- DataFrame Preview (qwerty layout) ---


...


--- AUTO-GENERATED TEST CODE ---
# these
df_real = load_real_data(['101030_1102308_0_0'])
tester_is_syllable_start.check_real(df_real, expected="10100", start_idx=21, layout='qwerty')
tester_is_syllable_start.check_real(df_real, expected="10100", start_idx=21, layout='dvorak')

tester_is_syllable_end.check_real(df_real, expected="01001", start_idx=21, layout='qwerty')
tester_is_syllable_end.check_real(df_real, expected="01001", start_idx=21, layout='dvorak')


Example 4: Found 'tomorrow' in Sequence: 101048_1102585_0_0
Start index: 17
Full Text preview: She has the game tomorrow...

--- DataFrame Preview (qwerty layout) ---


,17,18,19,20,21,22,23,24
key,t,o,m,o,r,r,o,w
is_syllable_start,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
is_syllable_end,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0



--- AUTO-GENERATED TEST CODE ---
# tomorrow
df_real = load_real_data(['101048_1102585_0_0'])
tester_is_syllable_start.check_real(df_real, expected="10010100", start_idx=17, layout='qwerty')
tester_is_syllable_start.check_real(df_real, expected="10010100", start_idx=17, layout='dvorak')

tester_is_syllable_end.check_real(df_real, expected="00101001", start_idx=17, layout='qwerty')
tester_is_syllable_end.check_real(df_real, expected="00101001", start_idx=17, layout='dvorak')


Example 5: Found 'tomorrow' in Sequence: 101581_1108889_0_0
Start index: 8
Full Text preview: Be back tomorrow ...

--- DataFrame Preview (qwerty layout) ---


,8,9,10,11,12,13,14,15
key,t,o,m,o,r,r,o,w
is_syllable_start,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
is_syllable_end,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0



--- AUTO-GENERATED TEST CODE ---
# tomorrow
df_real = load_real_data(['101581_1108889_0_0'])
tester_is_syllable_start.check_real(df_real, expected="10010100", start_idx=8, layout='qwerty')
tester_is_syllable_start.check_real(df_real, expected="10010100", start_idx=8, layout='dvorak')

tester_is_syllable_end.check_real(df_real, expected="00101001", start_idx=8, layout='qwerty')
tester_is_syllable_end.check_real(df_real, expected="00101001", start_idx=8, layout='dvorak')

